In [ ]:
import os
import requests
from openai import OpenAI, BadRequestError
from dotenv import load_dotenv
import gradio as gr
import json
from content_extractor import extract_text

In [3]:
load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [4]:
if GROQ_API_KEY is None or GROQ_API_KEY.strip() == "":
    raise ValueError("GROQ_API_KEY is not set in the environment variables.")
else:
    print("Groq API Key found")

Groq API Key found


In [5]:
openai = OpenAI(api_key=GROQ_API_KEY,base_url="https://api.groq.com/openai/v1")

In [6]:
SYSTEM_PROMPT = """
You are a snarky assistant who owns the restaurant.

Rules:
- When the user asks for the menu, available food, available dishes, or what is in stock, ALWAYS call the getFoodNames tool.
- Never guess or invent menu items.
- When the user asks for the price of a food item, ALWAYS call the getPriceOfFood tool.
- Never guess or invent prices.
- If you don't know something and no tool can provide the answer, say: "I don't know about it."
- After receiving tool results, use those results to answer the user naturally.
"""

In [7]:
url = "https://migrationology.com/pakistani-street-food-karachi/"
responseText = extract_text(url)

In [8]:
USER_PROMPT = f"Summarize the following text:' {responseText}'"

In [9]:
foods = [
    {
        "name": "Nihari",
        "category": "Breakfast",
        "price": 700
    },
    {
        "name": "Arabian Paratha",
        "category": "Breakfast",
        "price": 450
    },
    {
        "name": "Kulfi (Clay Cup)",
        "category": "Dessert",
        "price": 250
    },
    {
        "name": "Rabri",
        "category": "Dessert",
        "price": 350
    },
    {
        "name": "Bone Marrow Biryani",
        "category": "Main Course",
        "price": 1200
    },
    {
        "name": "Ninja Street Salad",
        "category": "Street Food",
        "price": 300
    },
    {
        "name": "Fish Kata-Kat",
        "category": "Main Course",
        "price": 900
    },
    {
        "name": "Bun Kebab",
        "category": "Street Food",
        "price": 180
    },
    {
        "name": "Chicken Curry",
        "category": "Main Course",
        "price": 650
    }
]

In [ ]:
def getPriceOfFood(food_name):
    for food in foods:
        if food["name"].lower() == food_name.lower():
            return f"the price of {food_name} is {food['price']} PKR."
    return f"{food_name} is not on the menu."

In [11]:
def getFoodNames(foods):
    return [food["name"] for food in foods]

In [12]:
price_function = {
    "name": "getPriceOfFood",
    "description": "Get the price of a food item from the collection of foods.",
    "parameters": {
        "type": "object",
        "properties": {
            "food_name": {
                "type": "string",
                "description": "The name of the food item to get the price for."
            }
        },
        "required": ["food_name"],
        "additionalProperties": False
    }
}

get_food_details = {
    "name": "getFoodNames",
    "description": "Get all the available food names currently on the menu.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

In [13]:
tools = [
    {
        "type": "function",
        "function": price_function
    },
    {
        "type": "function",
        "function": get_food_details
    }
]

In [14]:
def handle_tool_call(message):
    response = []
    for tools in message.tool_calls:
        if tools.function.name == "getPriceOfFood":
            arguments = json.loads(tools.function.arguments)
            food_name = arguments.get("food_name")
            result = getPriceOfFood(food_name)
            response.append({
                "role":"tool",
                "content":result,
                "tool_call_id":tools.id
            })
        elif tools.function.name == "getFoodNames":
            result = ", ".join(getFoodNames(foods))
            response.append({
                "role":"tool",
                "content":result,
                "tool_call_id":tools.id
            })
    print(response)
    return response;

In [ ]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":SYSTEM_PROMPT}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages,tools=tools)
    print(response)
    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        toolCallData = handle_tool_call(message)
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
                for tool_call in message.tool_calls
            ]
        })
        messages.extend(toolCallData)
        response = openai.chat.completions.create(model="llama-3.3-70b-versatile",messages=messages,tools=tools)

    return response.choices[0].message.content

In [16]:
gr.ChatInterface(fn=chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ChatCompletion(id='chatcmpl-6464738f-599d-4646-8c59-4258d07fd520', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='What can I do for you? Are you looking for a menu or something?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1786527487, model='llama-3.3-70b-versatile', object='chat.completion', moderation=None, service_tier='on_demand', system_fingerprint='fp_dae98b5ecb', usage=CompletionUsage(completion_tokens=17, prompt_tokens=430, total_tokens=447, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008266658, prompt_time=0.021429588, completion_time=0.079119113, total_time=0.100548701), usage_breakdown=None, x_groq={'id': 'req_01kztnc4xrempv2js0t6tmsanz', 'seed': 1742019506})
ChatCompletion(id='chatcmpl-9c4441f4-f181-4043-8e33-eda6f8aafba9', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(conten

Traceback (most recent call last):
  File "c:\Users\MuhammadFahd\Documents\projects\LLM-Engineering\llm-engineering-workspace\week1\text summarizer\.venv\Lib\site-packages\gradio\queueing.py", line 745, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\MuhammadFahd\Documents\projects\LLM-Engineering\llm-engineering-workspace\week1\text summarizer\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\MuhammadFahd\Documents\projects\LLM-Engineering\llm-engineering-workspace\week1\text summarizer\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\Users\MuhammadFahd\D